In [15]:
import pandas as pd

clientes = pd.read_csv("../data/raw/customers.csv")
polizas = pd.read_csv("../data/raw/policies.csv")
siniestros = pd.read_csv("../data/raw/claims.csv")
pagos = pd.read_csv("../data/raw/payments.csv")


In [16]:
polizas["producto"] = polizas["producto"].str.strip().str.lower()


mapa_producto = {
    "auto": "Auto",
    "automóvil": "Auto",
    "casa": "Hogar",
    "hogar": "Hogar",
    "seguro de vida": "Vida",
    "vida": "Vida",
    "gmm": "Gastos Médicos",
    "gastos medicos": "Gastos Médicos",
    "gastos médicos": "Gastos Médicos",
}

polizas["producto"] = polizas["producto"].map(mapa_producto)


In [17]:
polizas["producto"].value_counts(dropna = False)

producto
Auto              2133
Vida              1972
Gastos Médicos    1971
Hogar             1924
Name: count, dtype: int64

In [18]:
polizas["estatus"] = polizas["estatus"].str.strip().str.lower()

mapa_estatus_pol = {
    "active": "Activa", "activo": "Activa", "activa": "Activa", "vigente": "Activa",
    "lapsed": "Vencida", "vencida": "Vencida", "vencido": "Vencida",
    "cancelled": "Cancelada", "cancelada": "Cancelada", "cancelado": "Cancelada",
}

polizas["estatus"] = polizas["estatus"].map(mapa_estatus_pol)

polizas["estatus"].value_counts(dropna = False)

estatus
Activa       4885
Vencida      1953
Cancelada    1162
Name: count, dtype: int64

In [19]:
siniestros["estatus"] = siniestros["estatus"].str.strip().str.lower()

mapa_estatus_sin = {
    "paid": "Pagado", "pagado": "Pagado",
    "rejected": "Rechazado", "rechazado": "Rechazado",
    "approved": "Aprobado",
    "open": "Abierto",
    "en revisión": "En revisión",
}
siniestros["estatus"] = siniestros["estatus"].map(mapa_estatus_sin)
siniestros["estatus"].value_counts(dropna=False)

estatus
Pagado         1178
Rechazado      1114
En revisión     585
Abierto         570
Aprobado        553
Name: count, dtype: int64

In [20]:
pagos["metodo"] = pagos["metodo"].str.strip().str.lower()

mapa_metodo = {
    "tarjeta": "Tarjeta",
    "transferencia": "Transferencia", "spei": "Transferencia",
    "domiciliación": "Domiciliación",
    "oxxo": "OXXO",
}
pagos["metodo"] = pagos["metodo"].map(mapa_metodo)
pagos["metodo"].value_counts(dropna=False)

metodo
Tarjeta          5726
Transferencia    5722
NaN              2903
OXXO             2877
Domiciliación    2772
Name: count, dtype: int64

In [21]:
# polizas["fecha_inicio"] = pd.to_datetime(polizas["fecha_inicio"], format="mixed", dayfirst=True, errors="coerce")
# polizas["fecha_fin"] = pd.to_datetime(polizas["fecha_fin"], format="mixed", dayfirst=True, errors="coerce")

def a_fecha(serie):
    s = serie.astype(str).str.strip()
    f1 = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce") # 2022-06-07
    f2 = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce") # 27/12/1964
    f3 = pd.to_datetime(s, format="%m-%d-%Y", errors="coerce") # 12-23-2021

    return f1.fillna(f2).fillna(f3)

clientes["fecha_nacimiento"] = a_fecha(clientes["fecha_nacimiento"])
clientes["fecha_alta"]       = a_fecha(clientes["fecha_alta"])
polizas["fecha_inicio"]      = a_fecha(polizas["fecha_inicio"])
polizas["fecha_fin"]         = a_fecha(polizas["fecha_fin"])
siniestros["fecha_siniestro"]= a_fecha(siniestros["fecha_siniestro"])
pagos["fecha_pago"]          = a_fecha(pagos["fecha_pago"])


In [22]:
polizas[["fecha_inicio", "fecha_fin"]].dtypes

fecha_inicio    datetime64[us]
fecha_fin       datetime64[us]
dtype: object

In [23]:
polizas[["fecha_inicio","fecha_fin"]].head()

,fecha_inicio,fecha_fin
0,2026-08-25,2027-08-25
1,2024-01-11,2025-01-10
2,2022-06-07,2023-06-07
3,2023-01-04,2024-01-04
4,2022-01-05,2023-01-05


In [24]:
polizas["fecha_inicio"].isnull().sum()

np.int64(0)

In [33]:
print("Prima Negativas:", (polizas["prima_mxn"] < 0).sum())
print("Prima Nula:", (polizas["prima_mxn"].isnull().sum()))
print("Estadisticas de prima_mxn:")
print(polizas["prima_mxn"].describe())

Prima Negativas: 147
Prima Nula: 163
Estadisticas de prima_mxn:
count     7837.000000
mean     29753.771914
std      18693.679095
min     -59398.860000
25%      15344.010000
50%      30279.460000
75%      45127.920000
max      59987.070000
Name: prima_mxn, dtype: float64


In [36]:
polizas["prima_mxn"] = polizas["prima_mxn"].abs()

print("Prima Negativas:", (polizas["prima_mxn"] < 0).sum())
print(polizas["prima_mxn"].describe())

Prima Negativas: 0
count     7837.000000
mean     30897.717768
std      16735.075848
min       2004.610000
25%      16296.070000
50%      30808.640000
75%      45435.320000
max      59987.070000
Name: prima_mxn, dtype: float64


In [37]:
print("CLIENTES: ", clientes.isnull().sum(), "\n" )
print("POLIZAS: ", polizas.isnull().sum(), "\n" )
print("SINIESTROS: ", siniestros.isnull().sum(), "\n" )
print("PAGOS: ", pagos.isnull().sum(), "\n" )

CLIENTES:  customer_id           0
nombre                0
email               239
fecha_nacimiento      0
genero              729
ciudad                0
estado                0
fecha_alta            0
dtype: int64 

POLIZAS:  policy_id         0
customer_id       0
producto          0
fecha_inicio      0
fecha_fin         0
prima_mxn       163
estatus           0
dtype: int64 

SINIESTROS:  claim_id             0
policy_id            0
fecha_siniestro      0
monto_mxn          180
estatus              0
categoria          537
dtype: int64 

PAGOS:  payment_id       0
policy_id        0
fecha_pago       0
monto_mxn        0
metodo        2903
dtype: int64 



In [39]:
clientes["email"] = clientes["email"].fillna("Desconocido")
clientes["genero"] = clientes["genero"].fillna("Desconocido")
siniestros["categoria"] = siniestros["categoria"].fillna("Sin categoría")
pagos["metodo"] = pagos["metodo"].fillna("Desconocido")


In [40]:
print("Nulos email:", clientes["email"].isnull().sum())
print("Nulos genero:", clientes["genero"].isnull().sum())
print("Nulos categoria:", siniestros["categoria"].isnull().sum())
print("Nulos metodo:", pagos["metodo"].isnull().sum())
print("Nulos prima_mxn (a propósito):", polizas["prima_mxn"].isnull().sum())

Nulos email: 0
Nulos genero: 0
Nulos categoria: 0
Nulos metodo: 0
Nulos prima_mxn (a propósito): 163


In [42]:
print("Filas duplicadas en clientes:", clientes.duplicated().sum())

Filas duplicadas en clientes: 40


In [51]:
antes = len(clientes)
clientes = clientes.drop_duplicates()

print(f"Antes: {antes} , Despues: {len(clientes)}")

Antes: 5040 , Despues: 5000


In [55]:
clientes["ciudad"] = clientes["ciudad"].str.strip().str.title()
clientes["ciudad"].value_counts(dropna=False).head(15)

ciudad
Nueva Bangladesh                         18
Vieja Belarús                            14
Vieja Sudáfrica                          14
Nueva Nueva Zelandia                     14
Vieja Mongolia                           14
Nueva Barbados                           13
Vieja República Democrática Del Congo    13
Vieja Georgia                            13
Vieja Somalia                            13
Nueva Islandia                           13
Nueva Timor-Leste                        13
Vieja Congo                              13
Vieja Mozambique                         13
Nueva República Dominicana               13
Vieja Chipre                             12
Name: count, dtype: int64